# P.R.I.S.M. - Kaggle Evaluation Notebook (llama.cpp version)

This notebook is designed for judges and evaluators to test the P.R.I.S.M. (Probabilistic Reasoning and Interpretability System for Models) interface and capabilities directly from the browser, running the optimized MXFP4 model natively via llama.cpp.

### Instructions:
1. Ensure your Kaggle notebook has the **T4 x2** accelerator enabled in the Session Options.
2. Ensure **Internet** is toggled **On**.
3. Run all the cells below in order.
4. The final cell will generate a public URL. Click it to access the P.R.I.S.M. Glass Box UI.
5. *Note: If LocalTunnel prompts you for an "Endpoint IP", copy and paste the IP address printed right above the link.*

In [ ]:
# 1. Setup Environment & Clone Repository
!echo "Installing Node.js..."
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

!echo "\nCloning P.R.I.S.M. repository..."
!git clone https://github.com/chandan989/P.R.I.S.M..git

In [ ]:
# 2. Install llama-cpp-python and Download Model
!echo "Installing llama-cpp-python with CUDA 12.1 support..."
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 huggingface_hub

!echo "\nDownloading P.R.I.S.M. MXFP4 GGUF model..."
!mkdir -p models
!huggingface-cli download chandan989/gemma-4-26B-A4B-it-MXFP4_MOE --local-dir models --include "*.gguf"

In [ ]:
# 3. Configure and Start the FastAPI Backend
import subprocess
import time
import os

# Configure .env to use llama_cpp instead of ollama
env_path = "P.R.I.S.M./backend/.env"
!cp P.R.I.S.M./backend/.env.example {env_path}
!sed -i 's|MODEL_BACKEND=ollama|MODEL_BACKEND=llama_cpp|g' {env_path}
!sed -i 's|MODEL_PATH=|MODEL_PATH=../../models/|g' {env_path}
!sed -i 's|KB_ROOT=./knowledge_base|KB_ROOT=../knowledge_base|g' {env_path}

print("Installing Python backend dependencies...")
!pip install -r P.R.I.S.M./backend/requirements.txt

print("\nStarting FastAPI backend server on port 8000 (Model will load into VRAM now, please wait)...\n")
backend_process = subprocess.Popen(
    ["python3", "server.py", "--port", "8000"],
    cwd="P.R.I.S.M./backend"
)
time.sleep(15) # Wait for backend to initialize and model to load into VRAM

In [ ]:
# 4. Start Frontend and Expose via LocalTunnel
import urllib.request

print("Installing Node modules for the frontend...")
!cd P.R.I.S.M./prism-web && npm install

print("\nStarting Vite frontend...")
frontend_process = subprocess.Popen(
    ["npm", "run", "dev", "--", "--host", "0.0.0.0"],
    cwd="P.R.I.S.M./prism-web"
)
time.sleep(5)

print("\n" + "="*60)
print("🚀 SYSTEM READY: GENERATING PUBLIC URL")
print("="*60)

ipv4 = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"\n\033[1mIMPORTANT:\033[0m When you click the link below, you may be asked to enter a Tunnel Password.")
print(f"Your Tunnel Password / Endpoint IP is: \033[1;32m{ipv4}\033[0m\n")

# Expose the Vite default port (5173)
!npx localtunnel --port 5173